Full pipeline: land cover -> conductance, GeoMAD/MAD, CHELSA bioclim, and
the ensemble species distribution model, all in one run.



In [1]:
#!pip install elapid pygam

In [2]:
import os
import sys
import time
import glob
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import rasterio
import rasterio.windows
from rasterio.io import MemoryFile
from rasterio.mask import mask as rio_mask
from rasterio.merge import merge as rio_merge
from rasterio.transform import from_bounds
from rasterio.warp import Resampling, reproject
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import xgboost as xgb
from pygam import LogisticGAM, s, l
import elapid
import cartopy.io.shapereader as shpreader
import json
import os

import dask
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
import toolz
import xarray as xr
import datacube
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.dask import create_local_dask_cluster
from deafrica_tools.datahandling import load_ard
#from dekad import group_by_dekad, bin_by_interest_period, get_dekad_no_in_year, get_no_data_mask, max_consecutive_ones
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
from deafrica_tools.plotting import rgb
from deafrica_tools.app import geomedian
from odc.geo.xr import rasterize
from pyarrow import Table
from pyarrow.parquet import write_table




def progress(*args):
    print(f"[{datetime.now():%H:%M:%S}] " + " ".join(str(a) for a in args), flush=True)

In [15]:

RNG_SEED = 123
np.random.seed(RNG_SEED)

BIO_DIR = "/home/jovyan/insects/deafrica_insect/CHELSA/"
OCC_PATH = "data/H_rufipes_Africa_clean_unique.csv"
SPECIES_NAME = "Hyalomma_rufipes"

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [4]:
# Connect to the datacube
dc = datacube.Datacube(app="InsectDistributionModelling")

### Set up a Dask cluster

Dask can be used to better manage memory use and conduct the analysis in parallel. 
For an introduction to using Dask with Digital Earth Africa, see the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

>**Note**: We recommend opening the Dask processing window to view the different computations that are being executed; to do this, see the *Dask dashboard in DE Africa* section of the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

To use Dask, set up the local computing cluster using the cell below.

In [5]:
create_local_dask_cluster()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/leila.maritim@digitalearthafrica.org/proxy/8787/status,
Dashboard: /user/leila.maritim@digitalearthafrica.org/proxy/8787/status,Workers: 1
Total threads: 62,Total memory: 477.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43173,Workers: 0
Dashboard: /user/leila.maritim@digitalearthafrica.org/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38911,Total threads: 62
Dashboard: /user/leila.maritim@digitalearthafrica.org/proxy/37605/status,Memory: 477.21 GiB
Nanny: tcp://127.0.0.1:33405,


### Analysis parameters
This section defines the analysis parameters, including:

* `lat, lon, buffer`: center lat/lon and analysis window size for the area of interest
* `resolution`: the pixel resolution to use for loading the LULC dataset. The native resolution of the product is 10 metres i.e. `(-10,10)`
* `measurements`: the 'band' or measurement to load from the product, we can use the native measurement names of one of the aliases

In [6]:
resolution = (-5000, 5000)
output_crs = "EPSG:6933"
dask_chunks = dict(x=100,y=100)
output_dir = "results"


# Create the outpur directory if it does not exist.
os.makedirs(output_dir, exist_ok=True)

# Create the output folder for the selected country
output_path = os.path.join(output_dir, "ZA.parquet")

In [7]:
# Read GADM South Africa level 1 file
geopolygon = gpd.read_file('data/gadm41_ZAF.gpkg', layer='ADM_ADM_0')
# convert to an ODC Geometry object, keeping the source CRS
geom = Geometry(geopolygon.geometry.iloc[0], crs=geopolygon.crs)

### Load ESA LandCover data
Load the Landsat surface reflectance data from the datacube using the analysis parameters set in the previous section 

In [8]:
resolution=(-500, 500) #resample so we can view a large area

measurements='classification'


query = {
    'geopolygon': geom,
    'resolution': resolution,
    'measurements': measurements,
    'output_crs': 'EPSG:6933',  # or whatever CRS your product uses
    'group_by': 'solar_day',
}

ds = dc.load(product='esa_worldcover_2021', time='2021',**query).squeeze()
print(ds)

<xarray.Dataset> Size: 9MB
Dimensions:         (y: 2854, x: 3174)
Coordinates:
  * y               (y) float64 23kB -2.755e+06 -2.755e+06 ... -4.181e+06
  * x               (x) float64 25kB 1.587e+06 1.588e+06 ... 3.173e+06 3.174e+06
    time            datetime64[ns] 8B 2021-07-02
    spatial_ref     int32 4B 6933
Data variables:
    classification  (y, x) uint8 9MB 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref


## 2. LOAD AND CLEAN OCCURRENCE DATA

In [9]:
occ_raw = pd.read_csv(OCC_PATH)
lon_col = next(c for c in occ_raw.columns if c.lower() in {"lon", "longitude", "decimallongitude", "x"})
lat_col = next(c for c in occ_raw.columns if c.lower() in {"lat", "latitude", "decimallatitude", "y"})
occ_all = occ_raw.rename(columns={lon_col: "lon", lat_col: "lat"}).drop_duplicates().dropna(subset=["lon", "lat"])
occ_clean = occ_all[(occ_all.lon.between(-180, 180)) & (occ_all.lat.between(-90, 90))].copy()

occ_sf = gpd.GeoDataFrame(occ_clean, geometry=gpd.points_from_xy(occ_clean.lon, occ_clean.lat), crs="EPSG:4326")
inside = gpd.sjoin(occ_sf, geopolygon[["geometry"]], how="left", predicate="intersects")
occ = occ_clean[~inside["index_right"].isna().to_numpy()][["lon", "lat"]].drop_duplicates().reset_index(drop=True)
progress(f"Final modelling records: {len(occ)}")

[19:38:03] Final modelling records: 610


## 3. LAND COVER -> CONDUCTANCE

In [10]:
CONDUCTANCE_YEAR = 2021
CONDUCTANCE_RES_DEG = 0.01
VALID_LC_CLASSES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
def compute_conductance(lc_array):
    """Reclassifies land cover into 6 dry/wet-season conductance arrays
    with Monte Carlo uncertainty - identical logic/values to
    Conductance_Komi.R / conductance_komi.py, just returning arrays
    instead of writing rasters."""
    conductance_table = pd.DataFrame({
        "class": [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
        "dry_mean": [0.50, 0.50, 1.00, 1.00, 0.125, 0.75, 0.00, 0.50, 0.50, 0.50, 0.50],
        "wet_mean": [0.50, 0.50, 1.00, 0.125, 0.125, 0.75, 0.00, 0.25, 0.25, 0.25, 0.50],
    })
    conductance_table["dry_low"] = (conductance_table["dry_mean"] - 0.25).clip(lower=0)
    conductance_table["dry_high"] = (conductance_table["dry_mean"] + 0.25).clip(upper=1)
    conductance_table["wet_low"] = (conductance_table["wet_mean"] - 0.25).clip(lower=0)
    conductance_table["wet_high"] = (conductance_table["wet_mean"] + 0.25).clip(upper=1)
    for season in ("dry", "wet"):
        zero_mask = conductance_table[f"{season}_mean"] == 0
        conductance_table.loc[zero_mask, f"{season}_low"] = 0
        conductance_table.loc[zero_mask, f"{season}_high"] = 0
        one_mask = conductance_table[f"{season}_mean"] == 1
        conductance_table.loc[one_mask, f"{season}_low"] = 0.80
        conductance_table.loc[one_mask, f"{season}_high"] = 1.00

    rng = np.random.default_rng(123)
    n_mc = 2000
    progress(f"Running conductance Monte Carlo ({n_mc} iterations)...")
    dry_mc = rng.uniform(conductance_table["dry_low"].to_numpy(), conductance_table["dry_high"].to_numpy(), size=(n_mc, len(conductance_table)))
    wet_mc = rng.uniform(conductance_table["wet_low"].to_numpy(), conductance_table["wet_high"].to_numpy(), size=(n_mc, len(conductance_table)))

    conductance_table["dry_mc_mean"] = dry_mc.mean(axis=0)
    conductance_table["dry_mc_lower"] = np.quantile(dry_mc, 0.025, axis=0)
    conductance_table["dry_mc_upper"] = np.quantile(dry_mc, 0.975, axis=0)
    conductance_table["wet_mc_mean"] = wet_mc.mean(axis=0)
    conductance_table["wet_mc_lower"] = np.quantile(wet_mc, 0.025, axis=0)
    conductance_table["wet_mc_upper"] = np.quantile(wet_mc, 0.975, axis=0)

    def make_conductance(classes, values):
        lookup = np.full(int(max(classes)) + 1, np.nan, dtype=np.float32)
        for c, v in zip(classes, values):
            lookup[int(c)] = v
        out = np.full(lc_array.shape, np.nan, dtype=np.float32)
        valid = (lc_array >= 0) & (lc_array < len(lookup))
        out[valid] = lookup[lc_array[valid].astype(int)]
        return out

    classes = conductance_table["class"].to_numpy()
    return {
        "dry_mean": make_conductance(classes, conductance_table["dry_mc_mean"]),
        "dry_lower": make_conductance(classes, conductance_table["dry_mc_lower"]),
        "dry_upper": make_conductance(classes, conductance_table["dry_mc_upper"]),
        "wet_mean": make_conductance(classes, conductance_table["wet_mc_mean"]),
        "wet_lower": make_conductance(classes, conductance_table["wet_mc_lower"]),
        "wet_upper": make_conductance(classes, conductance_table["wet_mc_upper"]),
    }
lc_array = ds['classification'].values

conductance_raw = compute_conductance(lc_array)
progress("Conductance computed (6 layers).")


[19:38:03] Running conductance Monte Carlo (2000 iterations)...
[19:38:04] Conductance computed (6 layers).


## 4. GEOMAD/MAD 

## 4. GEOMAD/MAD 
## Background

Satellite imagery allows us to observe the Earth with significant accuracy and detail. However, missing data &mdash; such as gaps caused by cloud cover &mdash; can make it difficult to put together a complete image. 

In order to produce a single, complete view of a certain area, satellite data must be consolidated, stacking measurements from different points in time to create a composite image.

The Digital Earth Africa (DE Africa) GeoMAD (**Geo**median and **M**edian **A**bsolute **D**eviations) is a cloud-free composite of satellite data compiled for over annual and semi-annual (six-month) periods during each calendar year.

The following GeoMAD products are available on DE Africa platforms:

* `gm_s2_annual`: Annual (calendar year) GeoMAD composite using Sentinel-2 imagery, available for the years **2017 - present**
* `gm_s2_semiannual`: bi-annual (Jan-Jun, Jul-Dec) GeoMAD composites using Sentinel-2 imagery, available for the years **2017 - present**
* `gm_ls8_ls9_annual`: Annual (calendar year) GeoMAD composite using Landsat-8 and Landsat-9 imagery, available for the years **2021 - present**
* `gm_ls8_annual`: Annual (calendar year) GeoMAD composite using Landsat-8 imagery, available for the years **2013 - 2020**
* `gm_ls5_ls7_annual`: Annual (calendar year) GeoMAD composite combining both Landsat-5 and Landsat-7 imagery, available for the years **1984 - 2012**

Each product combines measurements collected over a defined period (annual or semi-annual) to produce one representative, multi-spectral image for every pixel of the African continent. The end result is a comprehensive dataset that can be used either to generate true-colour images for visual inspection of the landsacpe, or the full spectral dataset can be used to develop more complex algorithms.  

To produce a GeoMAD composite, for each pixel, invalid data is discarded, and the remaining observations are mathematically summarised using a high-dimensional geomedian statistic. GeoMAD also includes three measures of Median Absolute Deviation (MAD). These are higher-order statistical measurements on variation relative to the geomedian, pre-calculated at the same annual time scale. These layers can be used on their own or together with geomedian to gain insights about the land surface and understand its change over time. For a detailed description on how the GeoMAD is calculated, see the [GeoMAD technical specifications](https://docs.digitalearthafrica.org/en/latest/data_specs/GeoMAD_specs.html).

**Important details:**

* Datacube product names: `gm_s2_annual`, `gm_s2_semiannual`, `gm_ls8_ls9_annual`, `gm_ls8_annual`, `gm_ls5_ls7_annual`
* Geomedian surface reflectance product
    * Valid scaling range: `1 - 10,000`
    * `0` is `no data`
* Median Absolute Deviation product
    * Valid scaling range: Spectral MAD: `0 - 1` , Bray-Curtis MAD `0 - 1`, Euclidean MAD `0 - 10,000` 
    * `NaN` is `nodata`
* Status: Operational
* Date-range: 1984 &ndash; present
* Spatial resolution: 10m for S2 products, 30m for Landsat products

>Note: For a detailed description of DE Africa's GeoMAD service, see the DE Africa [GeoMAD technical specifications](https://docs.digitalearthafrica.org/en/latest/data_specs/GeoMAD_specs.html).

In [12]:
GEOMAD_YEAR = 2021

query = {
    'geopolygon': geom,
    'resolution': resolution,
    'output_crs': 'EPSG:6933',  
} 


ds_geomad = dc.load(
    product="gm_ls8_ls9_annual",
    time=str(GEOMAD_YEAR),
    measurements=['red', 'nir', 'swir_1', 'SMAD', 'EMAD', 'BCMAD'],
    **query,
).squeeze()

red, nir, swir_1 = (ds_geomad[b].values.astype("float32") for b in ("red", "nir", "swir_1"))
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (nir - red) / (nir + red)
    ndmi = (nir - swir_1) / (nir + swir_1)

geomad_raw = {
    "geomad_ndvi": ndvi,
    "geomad_ndmi": ndmi,
    "geomad_emad": ds_geomad["EMAD"].values.astype("float32"),
    "geomad_smad": ds_geomad["SMAD"].values.astype("float32"),
    "geomad_bcmad": ds_geomad["BCMAD"].values.astype("float32"),
}

geomad_transform = ds_geomad.odc.geobox.affine
geomad_crs = query['output_crs']  # "EPSG:6933" — matches the load extent

progress("GeoMAD/MAD computed (5 layers) via dc.load().")

[20:04:42] GeoMAD/MAD computed (5 layers) via dc.load().


/tmp/ipykernel_697/3074891509.py:30: DeprecationWarning: Geobox extraction logic has moved to odc-geo and the .geobox property is now deprecated. Please access via .odc.geobox instead.
  geomad_transform = ds_geomad.geobox.affine


## 5. LOAD CHELSA BIOCLIM

In [21]:
lc_transform = ds.odc.geobox.affine
lc_crs = ds.odc.geobox.crs
full_bounds = geopolygon.to_crs(src_crs).total_bounds
bio_files = sorted(
    glob.glob(os.path.join(BIO_DIR, "*.tif")),
    key=lambda f: int(re.search(r"bio_?(\d+)", os.path.basename(f)).group(1)),
)
if not bio_files:
    raise RuntimeError("No .tif files found in BIO_DIR.")
FACT = 10

with rasterio.open(bio_files[0]) as ref:
    src_crs = ref.crs
    src_transform = ref.transform
    pad_deg = 0.5
    minx, miny, maxx, maxy = full_bounds
    window = rasterio.windows.from_bounds(
        minx - pad_deg, miny - pad_deg, maxx + pad_deg, maxy + pad_deg, transform=src_transform,
    ).round_lengths().round_offsets()

bio_agg_bands = []
for f in bio_files:
    with rasterio.open(f) as src:
        arr = src.read(1, window=window).astype("float32")
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        h, w = arr.shape
        th, tw = (h // FACT) * FACT, (w // FACT) * FACT
        arr = arr[:th, :tw]
        bio_agg_bands.append(np.nanmean(arr.reshape(th // FACT, FACT, tw // FACT, FACT), axis=(1, 3)).astype("float32"))

bio_agg = np.stack(bio_agg_bands)
bio_names = [f"bio{i+1}" for i in range(len(bio_files))]

window_transform = rasterio.windows.transform(window, src_transform)
agg_transform = rasterio.Affine(
    window_transform.a * FACT, window_transform.b, window_transform.c,
    window_transform.d, window_transform.e * FACT, window_transform.f,
)

bio_profile = {
    "driver": "GTiff", "height": bio_agg.shape[1], "width": bio_agg.shape[2],
    "count": bio_agg.shape[0], "dtype": "float32", "crs": src_crs, "transform": agg_transform,
}
with MemoryFile() as memfile:
    with memfile.open(**bio_profile) as tmp:
        tmp.write(bio_agg.astype("float32"))
    with memfile.open() as tmp:
        geom = geopolygon.to_crs(src_crs).geometry
        bio_crop_arr, bio_crop_transform = rio_mask(tmp, geom, crop=True)

grid_shape = bio_crop_arr.shape[1:]


def reproject_array_to_grid(arr, src_transform_, src_crs_):
    dest = np.full(grid_shape, np.nan, dtype="float32")
    reproject(
        source=arr.astype("float32"), destination=dest,
        src_transform=src_transform_, src_crs=src_crs_,
        dst_transform=bio_crop_transform, dst_crs=src_crs,
        resampling=Resampling.bilinear,
    )
    return dest


conductance_names = list(conductance_raw.keys())
conductance_arrs = {n: reproject_array_to_grid(a, geomad_transform, lc_crs) for n, a in conductance_raw.items()}

geomad_names = list(geomad_raw.keys())
geomad_arrs = {n: reproject_array_to_grid(a, geomad_transform, geomad_crs) for n, a in geomad_raw.items()}

env_crop_arr = np.concatenate([bio_crop_arr, np.stack(list(conductance_arrs.values())), np.stack(list(geomad_arrs.values()))])
env_names = bio_names + conductance_names + geomad_names
progress(f"Combined predictor pool: {len(env_names)} candidate variables (bioclim + conductance + GeoMAD/MAD).")

[20:23:27] Combined predictor pool: 30 candidate variables (bioclim + conductance + GeoMAD/MAD).


## 6. VARIABLE SELECTION: CORRELATION DENDROGRAM + VIF 

In [22]:
def raster_to_df(arr, transform, names):
    n, h, w = arr.shape
    rows, cols = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    xs, ys = rasterio.transform.xy(transform, rows.ravel(), cols.ravel())
    df = pd.DataFrame({"x": xs, "y": ys})
    for i, name in enumerate(names):
        df[name] = arr[i].ravel()
    return df


def extract_at_points(arr, transform, names, lon, lat):
    rows, cols = rasterio.transform.rowcol(transform, lon, lat)
    rows, cols = np.array(rows), np.array(cols)
    h, w = arr.shape[1], arr.shape[2]
    valid = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    out = pd.DataFrame(np.nan, index=range(len(lon)), columns=names)
    for i, name in enumerate(names):
        vals = np.full(len(lon), np.nan)
        vals[valid] = arr[i][rows[valid], cols[valid]]
        out[name] = vals
    return out


full_env_df = raster_to_df(env_crop_arr, bio_crop_transform, env_names).drop(columns=["x", "y"]).dropna()
bio_sample = full_env_df.sample(n=min(20000, len(full_env_df)), random_state=RNG_SEED)

cor_mat = bio_sample.corr(method="pearson")
dist_condensed = (1 - cor_mat.abs()).to_numpy(copy=True)
np.fill_diagonal(dist_condensed, 0)
hc = linkage(squareform(dist_condensed, checks=False), method="average")
cluster_ids = fcluster(hc, t=0.3, criterion="distance")
cluster_df = pd.DataFrame({"variable": cor_mat.columns, "cluster": cluster_ids})

selected_corr = []
for cl in cluster_df.cluster.unique():
    vars_in_cluster = cluster_df.variable[cluster_df.cluster == cl].tolist()
    if len(vars_in_cluster) == 1:
        selected_corr.append(vars_in_cluster[0])
    else:
        sub_cor = cor_mat.loc[vars_in_cluster, vars_in_cluster].abs()
        selected_corr.append(sub_cor.mean(axis=1).idxmin())


def vifstep(df, threshold=5.0):
    vars_left = list(df.columns)
    while len(vars_left) > 1:
        X = sm.add_constant(df[vars_left])
        vifs = pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=vars_left)
        if vifs.max() <= threshold:
            break
        vars_left.remove(vifs.idxmax())
    return vars_left


vif_sample = full_env_df[selected_corr].sample(n=min(20000, len(full_env_df)), random_state=RNG_SEED)
selected_vars = vifstep(vif_sample, threshold=5.0)
progress(f"Final selected variables after VIF: {selected_vars}")

bio_sel_idx = [env_names.index(v) for v in selected_vars]
bio_sel_arr = env_crop_arr[bio_sel_idx]
bio_sel_names = selected_vars
bio_sel_transform = bio_crop_transform
bio_sel_crs = src_crs

[20:24:55] Final selected variables after VIF: ['bio15', 'bio18', 'bio14', 'bio19', 'dry_lower', 'geomad_ndvi', 'geomad_emad', 'geomad_smad']


## 7. BACKGROUND DESIGNS

In [23]:
N_BG = 10000


def valid_cell_coords(arr, transform):
    rows, cols = np.where(np.isfinite(arr))
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    return np.array(xs), np.array(ys)


valid_x, valid_y = valid_cell_coords(bio_sel_arr[0], bio_sel_transform)
rng = np.random.default_rng(RNG_SEED)
idx = rng.choice(len(valid_x), size=min(N_BG, len(valid_x)), replace=False)
bg_random = pd.DataFrame({"lon": valid_x[idx], "lat": valid_y[idx]})


def sample_kernel_bg(template_arr, transform, occ_df, n=10000, seed=RNG_SEED):
    from scipy.ndimage import convolve
    h, w = template_arr.shape
    occ_r = np.zeros((h, w), dtype="float64")
    rows, cols = rasterio.transform.rowcol(transform, occ_df.lon.to_numpy(), occ_df.lat.to_numpy())
    for r, c in zip(rows, cols):
        if 0 <= r < h and 0 <= c < w:
            occ_r[r, c] += 1
    dens = convolve(occ_r, np.ones((21, 21)), mode="constant", cval=0.0) + 1e-6
    dens[~np.isfinite(template_arr)] = np.nan
    flat = dens.ravel()
    valid = np.isfinite(flat)
    weights = flat[valid]
    weights = weights / weights.sum()
    rng_local = np.random.default_rng(seed)
    chosen = rng_local.choice(np.where(valid)[0], size=min(n, valid.sum()), replace=False, p=weights)
    rows_c, cols_c = np.unravel_index(chosen, dens.shape)
    xs, ys = rasterio.transform.xy(transform, rows_c, cols_c)
    return pd.DataFrame({"lon": xs, "lat": ys})


bg_kernel = sample_kernel_bg(bio_sel_arr[0], bio_sel_transform, occ, n=N_BG)


def sample_env_stratified_bg(pred_arr, transform, names, n=10000, n_clusters=50, seed=RNG_SEED):
    df = raster_to_df(pred_arr, transform, names).dropna()
    df = df.sample(n=min(100000, len(df)), random_state=seed)
    env_scaled = StandardScaler().fit_transform(df.drop(columns=["x", "y"]))
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10, max_iter=100).fit(env_scaled)
    df = df.copy()
    df["cluster"] = km.labels_
    n_per_cluster = int(np.ceil(n / n_clusters))
    parts = [g.sample(n=min(n_per_cluster, len(g)), random_state=seed) for _, g in df.groupby("cluster")]
    sampled = pd.concat(parts)
    if len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=seed)
    return sampled[["x", "y"]].rename(columns={"x": "lon", "y": "lat"})


bg_env = sample_env_stratified_bg(bio_sel_arr, bio_sel_transform, bio_sel_names, n=N_BG, n_clusters=50)
background_designs = {"Random": bg_random, "Kernel_bias": bg_kernel, "Environmental_stratified": bg_env}

## 8. MODEL FUNCTIONS, EVALUATION, CROSS-VALIDATION 

In [24]:
def make_dataset(occ_df, bg_df, pred_arr, pred_transform, pred_names):
    pres_vals = extract_at_points(pred_arr, pred_transform, pred_names, occ_df.lon, occ_df.lat)
    bg_vals = extract_at_points(pred_arr, pred_transform, pred_names, bg_df.lon, bg_df.lat)
    pres_dat = pres_vals.copy(); pres_dat.insert(0, "pa", 1)
    bg_dat = bg_vals.copy(); bg_dat.insert(0, "pa", 0)
    dat = pd.concat([pres_dat, bg_dat], ignore_index=True).dropna()
    dat["pa"] = dat["pa"].astype(int)
    return dat


def eval_model(obs, pred):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    keep = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[keep], pred[keep]
    auc_val = roc_auc_score(obs, pred) if len(np.unique(obs)) > 1 else np.nan

    thresholds = np.arange(0.01, 1.0, 0.01)
    tss_values = np.full(len(thresholds), np.nan)
    for i, th in enumerate(thresholds):
        bp = (pred >= th).astype(int)
        tp, tn = np.sum((bp == 1) & (obs == 1)), np.sum((bp == 0) & (obs == 0))
        fp, fn = np.sum((bp == 1) & (obs == 0)), np.sum((bp == 0) & (obs == 1))
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        tss_values[i] = sens + spec - 1

    best_th = thresholds[np.nanargmax(tss_values)]
    bp = (pred >= best_th).astype(int)
    tp, tn = np.sum((bp == 1) & (obs == 1)), np.sum((bp == 0) & (obs == 0))
    fp, fn = np.sum((bp == 1) & (obs == 0)), np.sum((bp == 0) & (obs == 1))
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    tss = sensitivity + specificity - 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else np.nan
    return pd.DataFrame([{"AUC": auc_val, "Threshold": best_th, "Accuracy": accuracy,
                           "Sensitivity": sensitivity, "Specificity": specificity, "TSS": tss, "F1": f1}])


def fit_glm(train):
    X = sm.add_constant(train.drop(columns=["pa"]))
    return sm.GLM(train["pa"], X, family=sm.families.Binomial()).fit()


def fit_gam(train):
    predictors = [c for c in train.columns if c != "pa"]
    X = train[predictors].to_numpy()
    y = train["pa"].to_numpy()
    terms = None
    for i, col in enumerate(predictors):
        n_unique = train[col].nunique()
        term = l(i) if n_unique <= 5 else s(i, n_splines=min(4, n_unique - 1) + 3)
        terms = term if terms is None else terms + term
    model = LogisticGAM(terms)
    model.fit(X, y)
    model._predictors = predictors
    return model


def fit_rf(train):
    X, y = train.drop(columns=["pa"]), train["pa"]
    model = RandomForestClassifier(n_estimators=500, random_state=RNG_SEED).fit(X, y)
    model._predictors = list(X.columns)
    return model


def fit_brt(train):
    X, y = train.drop(columns=["pa"]), train["pa"]
    model = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=150, max_depth=3,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    ).fit(X, y)
    model._predictors = list(X.columns)
    return model


def fit_maxnet(train):
    x, y = train.drop(columns=["pa"]), train["pa"]
    keep_cols = [c for c in x.columns if x[c].nunique() > 2]
    x = x[keep_cols]
    model = elapid.MaxentModel(feature_types=["linear", "quadratic", "product", "hinge"])
    model.fit(x, y)
    model._predictors = keep_cols
    return model


def fit_svm(train):
    X, y = train.drop(columns=["pa"]), train["pa"]
    model = SVC(kernel="rbf", probability=True, random_state=RNG_SEED).fit(X, y)
    model._predictors = list(X.columns)
    return model


def predict_model(model, newdata, algorithm):
    newdata = pd.DataFrame(newdata)
    if algorithm == "GLM":
        X = sm.add_constant(newdata, has_constant="add").reindex(columns=model.model.exog_names, fill_value=0)
        return np.asarray(model.predict(X))
    if algorithm == "GAM":
        return np.asarray(model.predict_proba(newdata[model._predictors].to_numpy()))
    if algorithm == "RF":
        return model.predict_proba(newdata[model._predictors])[:, list(model.classes_).index(1)]
    if algorithm == "BRT":
        return model.predict_proba(newdata[model._predictors])[:, 1]
    if algorithm == "MAXNET":
        needed = [c for c in model._predictors if c in newdata.columns]
        return np.asarray(model.predict(newdata[needed]))
    if algorithm == "SVM":
        return model.predict_proba(newdata[model._predictors])[:, list(model.classes_).index(1)]
    raise ValueError(algorithm)


def fit_one_model(dat, alg):
    return {"GLM": fit_glm, "GAM": fit_gam, "RF": fit_rf, "BRT": fit_brt, "MAXNET": fit_maxnet, "SVM": fit_svm}[alg](dat)


def create_folds(dat, k=5, seed=RNG_SEED):
    dat = dat.copy()
    dat["fold"] = -1
    rng_local = np.random.default_rng(seed)
    for pa_val in (0, 1):
        idx = dat.index[dat.pa == pa_val].to_numpy()
        folds = np.resize(np.arange(1, k + 1), len(idx))
        rng_local.shuffle(folds)
        dat.loc[idx, "fold"] = folds
    return dat


def run_cv_ensemble_only(dat, algorithms, k=5):
    dat = create_folds(dat, k=k)
    out = []
    for fold in range(1, k + 1):
        train = dat[dat.fold != fold].drop(columns=["fold"])
        test = dat[dat.fold == fold].drop(columns=["fold"])
        test_x = test.drop(columns=["pa"])
        fold_preds = {}
        for alg in algorithms:
            try:
                model = fit_one_model(train, alg)
                fold_preds[alg] = predict_model(model, test_x, alg)
            except Exception:
                continue
        if not fold_preds:
            continue
        pred_mat = np.column_stack(list(fold_preds.values()))
        met = eval_model(test.pa, np.nanmean(pred_mat, axis=1))
        met["Fold"] = fold
        met["Algorithm"] = "ENSEMBLE_MEAN"
        met["N_algorithms"] = pred_mat.shape[1]
        out.append(met)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def predict_ensemble(train_dat, test_x, algorithms):
    preds = {}
    for alg in algorithms:
        try:
            model = fit_one_model(train_dat, alg)
            preds[alg] = predict_model(model, test_x, alg)
        except Exception:
            continue
    if not preds:
        return None
    pred_mat = np.column_stack(list(preds.values()))
    return {"mean": np.nanmean(pred_mat, axis=1), "sd": np.nanstd(pred_mat, axis=1, ddof=1), "n_algorithms": pred_mat.shape[1]}

## 9. TREATMENTS AND ALGORITHMS

In [25]:
treatments = {
    "Climate_only": (bio_sel_arr, bio_sel_names),
    "Climate_dry_mean": (np.concatenate([bio_sel_arr, conductance_arrs["dry_mean"][None]]), bio_sel_names + ["dry_mean"]),
    "Climate_dry_lower": (np.concatenate([bio_sel_arr, conductance_arrs["dry_lower"][None]]), bio_sel_names + ["dry_lower"]),
    "Climate_dry_upper": (np.concatenate([bio_sel_arr, conductance_arrs["dry_upper"][None]]), bio_sel_names + ["dry_upper"]),
    "Climate_wet_mean": (np.concatenate([bio_sel_arr, conductance_arrs["wet_mean"][None]]), bio_sel_names + ["wet_mean"]),
    "Climate_wet_lower": (np.concatenate([bio_sel_arr, conductance_arrs["wet_lower"][None]]), bio_sel_names + ["wet_lower"]),
    "Climate_wet_upper": (np.concatenate([bio_sel_arr, conductance_arrs["wet_upper"][None]]), bio_sel_names + ["wet_upper"]),
    "Climate_dry_wet_mean": (
        np.concatenate([bio_sel_arr, conductance_arrs["dry_mean"][None], conductance_arrs["wet_mean"][None]]),
        bio_sel_names + ["dry_mean", "wet_mean"],
    ),
}
algorithms = ["GLM", "GAM", "RF", "BRT", "MAXNET", "SVM"]

## 10. VALIDATION: 10% HOLDOUT + 5-FOLD CV 

In [26]:
rng777 = np.random.default_rng(777)
holdout_idx = rng777.choice(len(occ), size=int(np.ceil(0.10 * len(occ))), replace=False)
occ_holdout = occ.iloc[holdout_idx].reset_index(drop=True)
occ_train = occ.drop(index=holdout_idx).reset_index(drop=True)

holdout_results = []
for bg_name, bg_df in background_designs.items():
    for tr, (pred_arr, pred_names) in treatments.items():
        progress(f"10% holdout: {tr} | background: {bg_name}")
        train_dat = make_dataset(occ_train, bg_df, pred_arr, bio_sel_transform, pred_names)
        hold_vals = extract_at_points(pred_arr, bio_sel_transform, pred_names, occ_holdout.lon, occ_holdout.lat)
        bg_vals = extract_at_points(pred_arr, bio_sel_transform, pred_names, bg_df.lon, bg_df.lat)
        hold_dat = pd.concat([hold_vals.assign(pa=1), bg_vals.assign(pa=0)], ignore_index=True).dropna()
        ens = predict_ensemble(train_dat, hold_dat.drop(columns=["pa"]), algorithms)
        if ens is None:
            continue
        met = eval_model(hold_dat.pa, ens["mean"])
        met["Treatment"] = tr
        met["Background_design"] = bg_name
        holdout_results.append(met)
holdout_results = pd.concat(holdout_results, ignore_index=True)

all_cv = []
for bg_name, bg_df in background_designs.items():
    for tr, (pred_arr, pred_names) in treatments.items():
        progress(f"5-fold CV: {tr} | background: {bg_name}")
        dat = make_dataset(occ_train, bg_df, pred_arr, bio_sel_transform, pred_names)
        try:
            cv_metrics = run_cv_ensemble_only(dat, algorithms, k=5)
        except Exception as e:
            progress(f"ERROR in {tr} | {bg_name}: {e}")
            continue
        if cv_metrics.empty:
            continue
        cv_metrics["Treatment"] = tr
        cv_metrics["Background_design"] = bg_name
        all_cv.append(cv_metrics)
results_cv = pd.concat(all_cv, ignore_index=True)

summary_results = (
    results_cv.groupby(["Treatment", "Background_design"])
    .agg(AUC_mean=("AUC", "mean"), AUC_sd=("AUC", "std"), TSS_mean=("TSS", "mean"), TSS_sd=("TSS", "std"),
         F1_mean=("F1", "mean"))
    .reset_index()
    .sort_values("TSS_mean", ascending=False)
)

[20:25:48] 10% holdout: Climate_only | background: Random
[20:25:59] 10% holdout: Climate_dry_mean | background: Random
[20:26:13] 10% holdout: Climate_dry_lower | background: Random
[20:26:24] 10% holdout: Climate_dry_upper | background: Random


/opt/venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[20:26:38] 10% holdout: Climate_wet_mean | background: Random
[20:26:52] 10% holdout: Climate_wet_lower | background: Random
[20:27:06] 10% holdout: Climate_wet_upper | background: Random
[20:27:21] 10% holdout: Climate_dry_wet_mean | background: Random
[20:27:35] 10% holdout: Climate_only | background: Kernel_bias
[20:27:54] 10% holdout: Climate_dry_mean | background: Kernel_bias
[20:28:19] 10% holdout: Climate_dry_lower | background: Kernel_bias
[20:28:40] 10% holdout: Climate_dry_upper | background: Kernel_bias


/opt/venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[20:29:06] 10% holdout: Climate_wet_mean | background: Kernel_bias
[20:29:31] 10% holdout: Climate_wet_lower | background: Kernel_bias
[20:29:56] 10% holdout: Climate_wet_upper | background: Kernel_bias
[20:30:22] 10% holdout: Climate_dry_wet_mean | background: Kernel_bias
[20:30:47] 10% holdout: Climate_only | background: Environmental_stratified
[20:31:01] 10% holdout: Climate_dry_mean | background: Environmental_stratified
[20:31:17] 10% holdout: Climate_dry_lower | background: Environmental_stratified
[20:31:30] 10% holdout: Climate_dry_upper | background: Environmental_stratified


/opt/venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


[20:31:47] 10% holdout: Climate_wet_mean | background: Environmental_stratified
[20:32:04] 10% holdout: Climate_wet_lower | background: Environmental_stratified
[20:32:21] 10% holdout: Climate_wet_upper | background: Environmental_stratified
[20:32:38] 10% holdout: Climate_dry_wet_mean | background: Environmental_stratified
[20:32:55] 5-fold CV: Climate_only | background: Random
[20:33:36] 5-fold CV: Climate_dry_mean | background: Random
[20:34:28] 5-fold CV: Climate_dry_lower | background: Random
[20:35:09] 5-fold CV: Climate_dry_upper | background: Random
[20:35:59] 5-fold CV: Climate_wet_mean | background: Random
[20:36:51] 5-fold CV: Climate_wet_lower | background: Random
[20:37:44] 5-fold CV: Climate_wet_upper | background: Random
[20:38:35] 5-fold CV: Climate_dry_wet_mean | background: Random
[20:39:27] 5-fold CV: Climate_only | background: Kernel_bias
[20:40:35] 5-fold CV: Climate_dry_mean | background: Kernel_bias
[20:42:06] 5-fold CV: Climate_dry_lower | background: Kernel_bia

## 11. BALANCED MODEL SELECTION (CV + holdout only - East Africa - transfer is skipped since occ is - South-Africa-only in this scope)

In [27]:
cv_rank = summary_results[["Treatment", "Background_design", "AUC_mean", "TSS_mean"]].rename(
    columns={"AUC_mean": "CV_AUC", "TSS_mean": "CV_TSS"})
cv_rank["CV_rank"] = cv_rank["CV_TSS"].rank(ascending=False, method="dense")

holdout_rank = holdout_results[["Treatment", "Background_design", "AUC", "TSS"]].rename(
    columns={"AUC": "Holdout_AUC", "TSS": "Holdout_TSS"})
holdout_rank["Holdout_rank"] = holdout_rank["Holdout_TSS"].rank(ascending=False, method="dense")

balanced_selection = cv_rank.merge(holdout_rank, on=["Treatment", "Background_design"])
balanced_selection["Mean_rank"] = balanced_selection[["CV_rank", "Holdout_rank"]].mean(axis=1)
balanced_selection = balanced_selection.sort_values(["Mean_rank", "CV_TSS", "Holdout_TSS"], ascending=[True, False, False])

best_row = balanced_selection.iloc[0]
best_treatment, best_background = best_row["Treatment"], best_row["Background_design"]
progress(f"Best balanced treatment: {best_treatment}")
progress(f"Best balanced background design: {best_background}")

[20:58:41] Best balanced treatment: Climate_wet_mean
[20:58:41] Best balanced background design: Environmental_stratified


## 12. FINAL PREDICTION 

In [29]:
pred_arr, pred_names = treatments[best_treatment]
bg_df = background_designs[best_background]
dat = make_dataset(occ, bg_df, pred_arr, bio_sel_transform, pred_names)


def predict_raster_model(model, algorithm, pred_arr, pred_names):
    n, h, w = pred_arr.shape
    flat = pred_arr.reshape(n, -1).T
    df = pd.DataFrame(flat, columns=pred_names)
    out = np.full(flat.shape[0], np.nan)
    keep = df.notna().all(axis=1).to_numpy()
    if keep.sum() > 0:
        try:
            out[keep] = predict_model(model, df.loc[keep], algorithm)
        except Exception:
            pass
    return out.reshape(h, w)


model_maps = {}
for alg in algorithms:
    progress(f"Mapping algorithm: {alg}")
    try:
        model = fit_one_model(dat, alg)
        model_maps[alg] = predict_raster_model(model, alg, pred_arr, pred_names)
    except Exception as e:
        progress(f"  {alg} failed: {e}")

if len(model_maps) < 2:
    raise RuntimeError("Fewer than two algorithm maps were produced. Ensemble cannot be built.")

valid_algs = list(model_maps.keys())
alg_weight_rows = summary_results[
    (summary_results.Treatment == best_treatment) & (summary_results.Background_design == best_background)
]
# summary_results only ever has ensemble-level rows (see main_model_rufipes.py's
# note on this same gap) - always falls back to equal weighting here.
warnings.warn(f"No per-algorithm TSS available - equal weighting across: {', '.join(valid_algs)}")
alg_weights = np.ones(len(valid_algs))
alg_weights = alg_weights / alg_weights.sum()

stack = np.stack([model_maps[a] for a in valid_algs])
w = alg_weights[:, None, None]
finite = np.isfinite(stack)
w_masked = np.where(finite, np.broadcast_to(w, stack.shape), 0)
stack_filled = np.where(finite, stack, 0)
weight_sum = w_masked.sum(axis=0)
ens_mean = np.divide((stack_filled * w_masked).sum(axis=0), weight_sum, out=np.full(weight_sum.shape, np.nan), where=weight_sum > 0)

n_finite = finite.sum(axis=0)
sq_dev = np.where(finite, (stack - np.broadcast_to(ens_mean, stack.shape)) ** 2, 0)
ens_var = np.divide((w_masked * sq_dev).sum(axis=0), weight_sum, out=np.full(weight_sum.shape, np.nan), where=weight_sum > 0)
ens_var[n_finite < 2] = np.nan
ens_sd = np.sqrt(ens_var)
ens_lower95 = np.clip(ens_mean - 1.96 * ens_sd, 0, 1)
ens_upper95 = np.clip(ens_mean + 1.96 * ens_sd, 0, 1)

train_x = dat.drop(columns=["pa"])
mins, maxs = train_x.min(), train_x.max()
risk = np.zeros(pred_arr.shape[1:], dtype="float32")
for i, name in enumerate(pred_names):
    band = pred_arr[i]
    risk += ((band < mins[name]) | (band > maxs[name])).astype("float32")


def write_single_band(arr, name):
    profile = {
        "driver": "GTiff", "height": arr.shape[0], "width": arr.shape[1], "count": 1,
        "dtype": "float32", "crs": bio_sel_crs, "transform": bio_sel_transform, "nodata": np.nan,
    }
    with rasterio.open(os.path.join(output_dir, f"{SPECIES_NAME}_{name}.tif"), "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)


write_single_band(ens_mean, "BEST_ENSEMBLE_MEAN")
write_single_band(ens_sd, "BEST_ENSEMBLE_SD_UNCERTAINTY")
write_single_band(ens_lower95, "BEST_ENSEMBLE_LOWER95")
write_single_band(ens_upper95, "BEST_ENSEMBLE_UPPER95")
write_single_band(risk, "BEST_EXTRAPOLATION_RISK")

pd.DataFrame([{
    "Best_treatment": best_treatment,
    "Best_background_design": best_background,
    "Selected_variables": ", ".join(bio_sel_names),
    "CV_AUC": best_row["CV_AUC"], "CV_TSS": best_row["CV_TSS"],
    "Holdout_AUC": best_row["Holdout_AUC"], "Holdout_TSS": best_row["Holdout_TSS"],
    "N_occurrence_records": len(occ),
}]).to_csv(os.path.join(output_dir, "final_results_summary.csv"), index=False)

progress(f"Done. Final results only, written to {output_dir}")

[07:26:11] Mapping algorithm: GLM
[07:26:11] Mapping algorithm: GAM
[07:26:12] Mapping algorithm: RF
[07:26:24] Mapping algorithm: BRT
[07:26:24] Mapping algorithm: MAXNET
[07:26:27] Mapping algorithm: SVM
[07:26:30] Done. Final results only, written to results


/tmp/ipykernel_697/796116635.py:38: UserWarning: No per-algorithm TSS available - equal weighting across: GLM, GAM, RF, BRT, MAXNET, SVM
  warnings.warn(f"No per-algorithm TSS available - equal weighting across: {', '.join(valid_algs)}")
